# OnomaCap Factor Conditioning — canonical Jamo + BF16

HTSAT→BART 구조를 유지하면서 audio와 6개 acoustic factor의 factor×time token에 독립적인 decoder cross-attention을 적용하고, 한국어 의성어를 `(초성 중성 [종성]?) *` 형식의 canonical Jamo 토큰으로 학습합니다. 초반에는 factor 경로만 학습하고, 이후 audio modality dropout을 점진적으로 낮춰 joint training으로 전환합니다. 이 노트북은 factor 추출, smoke test, 학습, 평가, teacher-forced 자모×factor attention 집계를 순서대로 실행합니다.

In [ ]:
# 의존성 설치
%pip install -q transformers==4.36.2 tokenizers==0.15.2 "huggingface-hub<1.0" torchlibrosa==0.1.0 librosa==0.10.2.post1 ruamel.yaml==0.17.40 gdown==5.2.0 "kagglehub>=0.3.12" pycocoevalcap==1.2 "scikit-learn>=1.4,<1.7" "pandas>=2.1,<2.4" matplotlib seaborn tqdm loguru warmup-scheduler gensim

# 저장소 clone 및 WavCaps overlay 적용
import shutil
import subprocess
from pathlib import Path

ONOMAHOW_REPO = "https://github.com/youhan200203/OnomaHoW.git"
ONOMAHOW_REF = "working"
WAVCAPS_REPO = "https://github.com/XinhaoMei/WavCaps.git"
WAVCAPS_COMMIT = "a5a9649ce305d7fe82cfcf5d6a4a12f03df9ef1e"
ANNOTATION_REPO = "https://github.com/jspirit01/sound-to-onomatopoeia.git"

ONOMAHOW_DIR = Path("/content/OnomaHoW")
WAVCAPS_DIR = Path("/content/WavCaps")
ANNOTATION_DIR = Path("/content/sound-to-onomatopoeia")

for transient_dir in (ONOMAHOW_DIR, WAVCAPS_DIR, ANNOTATION_DIR):
    if transient_dir.exists():
        shutil.rmtree(transient_dir)

subprocess.run(
    ["git", "clone", "-q", "--depth", "1", "--branch", ONOMAHOW_REF, ONOMAHOW_REPO, str(ONOMAHOW_DIR)],
    check=True,
)
subprocess.run(["git", "clone", "-q", WAVCAPS_REPO, str(WAVCAPS_DIR)], check=True)
subprocess.run(
    ["git", "-C", str(WAVCAPS_DIR), "checkout", "--detach", "-q", WAVCAPS_COMMIT],
    check=True,
)
subprocess.run(["git", "clone", "-q", "--depth", "1", ANNOTATION_REPO, str(ANNOTATION_DIR)], check=True)

overlay_root = ONOMAHOW_DIR / "wavcaps_patch"
overlay_files = sorted(overlay_root.rglob("*.py"))
assert overlay_files, "wavcaps_patch overlay가 비어 있습니다."
for source in overlay_files:
    destination = WAVCAPS_DIR / source.relative_to(overlay_root)
    destination.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(source, destination)
    assert source.read_bytes() == destination.read_bytes()

helper_destination = WAVCAPS_DIR / "captioning/tools/jamo_preprocessing.py"
config_destination = WAVCAPS_DIR / "captioning/settings/onomacap_jamo.yaml"
factor_config_destination = WAVCAPS_DIR / "captioning/settings/onomacap_jamo_factor.yaml"
shutil.copy2(ONOMAHOW_DIR / "jamo_preprocessing.py", helper_destination)
shutil.copy2(ONOMAHOW_DIR / "configs/onomacap_jamo.yaml", config_destination)
shutil.copy2(ONOMAHOW_DIR / "configs/onomacap_jamo_factor.yaml", factor_config_destination)

checked_out_commit = subprocess.check_output(
    ["git", "-C", str(WAVCAPS_DIR), "rev-parse", "HEAD"], text=True
).strip()
assert checked_out_commit == WAVCAPS_COMMIT
print(f"WavCaps {checked_out_commit} + {len(overlay_files)} overlay files")

In [ ]:
# 공통 설정
import csv
import gc
import json
import os
import random
import sys
import unicodedata

import kagglehub
import numpy as np
import pandas as pd
import torch
from google.colab import drive

if str(ONOMAHOW_DIR) not in sys.path:
    sys.path.insert(0, str(ONOMAHOW_DIR))

from jamo_preprocessing import (
    CHOSEONG,
    EXPECTED_LATIN_AUDIO,
    EXPECTED_OUTPUT_ROWS,
    JAMO_VOCAB,
    JONGSEONG,
    JUNGSEONG,
    hangul_caption_to_romanized,
    jamo_to_hangul_caption,
    load_or_create_split_manifest,
    normalize_romanized_caption,
    prepare_rows,
)

MODEL_SEED = 20
SPLIT_SEED = 20
EVAL_BEAM_SIZE = 3
BASE_JAMO_EXPERIMENT_NAME = (
    "onomacap_htsat_bart_jamo_bf16_sharedsplit_vocabmask_v1"
)
EXPERIMENT_NAME = (
    "onomacap_htsat_bart_jamo_factor_dual_cross_attn_"
    "curriculum_bf16_sharedsplit_vocabmask_v1"
)
SEED = MODEL_SEED

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(SEED)
drive.mount("/content/drive")

# model seed만 바꾸어도 동일한 Drive split을 쓰고,
# 같은 seed의 no-factor Jamo checkpoint를 warm-start하도록 경로를 갱신한다.
import ruamel.yaml as yaml
factor_yaml = yaml.YAML()
with factor_config_destination.open("r", encoding="utf-8") as stream:
    factor_notebook_config = factor_yaml.load(stream)
factor_notebook_config["seed"] = MODEL_SEED
factor_notebook_config.setdefault("evaluation", {})["beam_size"] = EVAL_BEAM_SIZE
factor_notebook_config["pretrain_path"] = str(
    Path("/content/drive/MyDrive/OnomaCap/checkpoints")
    / (
        f"{BASE_JAMO_EXPERIMENT_NAME}_lr_3e-05_"
        f"batch_16_seed_{MODEL_SEED}"
    )
    / "best_model.pt"
)
with factor_config_destination.open("w", encoding="utf-8") as stream:
    factor_yaml.dump(factor_notebook_config, stream)

In [ ]:
# 한국어 annotation 정리 및 canonical Jamo target 생성
audio_root = Path(kagglehub.dataset_download("buraktaci/firat-esc50"))
csv_path = ANNOTATION_DIR / "sound-to-onomatopoeia_annotation.csv"

with csv_path.open(encoding="utf-8-sig", newline="") as stream:
    prepared_rows, processing_report = prepare_rows(csv.DictReader(stream))

assert processing_report.output_rows == EXPECTED_OUTPUT_ROWS == 7_961
assert processing_report.latin_rows_excluded == (EXPECTED_LATIN_AUDIO,)
assert set(processing_report.duplicate_rows_excluded) == {"38_68.wav", "38_95.wav"}
df = pd.DataFrame(prepared_rows)

def audio_key(name):
    return unicodedata.normalize("NFKC", Path(str(name)).name).strip().casefold()

extensions = {".mp3", ".wav", ".flac", ".ogg", ".m4a"}
audio_files = {
    audio_key(path.name): path
    for path in audio_root.rglob("*")
    if path.suffix.lower() in extensions
}
df["audio_path"] = df["audio_file"].map(
    lambda name: str(audio_files.get(audio_key(name), ""))
)

jamo_columns = [f"candidate{index}_jamo" for index in range(1, 6)]
assert len(df) == 7_961
assert df["class"].nunique() == 41
assert not df["audio_file"].duplicated().any()
assert not df[jamo_columns].isna().any().any()
assert not df["audio_path"].eq("").any()
assert df[jamo_columns].map(lambda value: set(value.split()) <= set(JAMO_VOCAB)).all().all()

print(processing_report)
print(df.loc[:, ["audio_file", *jamo_columns]].head(2).to_string(index=False))

In [ ]:
# 6×T acoustic factor 추출: Drive에 정상 파일이 있으면 재사용
FACTOR_OUTPUT = Path("/content/drive/MyDrive/OnomaCap/features/onomacap_acoustic_factors.npz")
LOCAL_FACTOR_OUTPUT = Path("/content/onomacap_acoustic_factors.npz")

def validate_factor_archive(path):
    with np.load(path, allow_pickle=False) as archive:
        expected_files = df["audio_file"].astype(str).to_numpy()
        np.testing.assert_array_equal(archive["audio_files"], expected_files)
        lengths = archive["lengths"]
        offsets = archive["offsets"]
        factor_values = archive["factor_values"]
        assert factor_values.shape[0] == 6
        assert len(lengths) == len(expected_files)
        assert len(offsets) == len(expected_files) + 1
        assert np.array_equal(np.diff(offsets), lengths)
        assert offsets[0] == 0 and offsets[-1] == factor_values.shape[1]
        assert archive["summaries"].shape[0] == len(expected_files)
        return factor_values.shape, lengths.min(), lengths.max()

if FACTOR_OUTPUT.is_file():
    factor_info = validate_factor_archive(FACTOR_OUTPUT)
    print("Existing acoustic factors:", FACTOR_OUTPUT, factor_info)
else:
    extractor = ONOMAHOW_DIR / "extract_acoustic_factors.py"
    assert extractor.is_file(), f"Missing extractor: {extractor}"
    factor_audio_csv = Path("/content/onomacap_factor_audio_files.csv")
    df[["audio_file"]].to_csv(factor_audio_csv, index=False)
    subprocess.run(
        [
            sys.executable, str(extractor),
            "--annotation-csv", str(factor_audio_csv),
            "--audio-root", str(audio_root),
            "--output", str(LOCAL_FACTOR_OUTPUT),
            "--drive-output", str(FACTOR_OUTPUT),
            "--overwrite",
        ],
        check=True,
    )
    factor_info = validate_factor_archive(FACTOR_OUTPUT)
    print("Extracted acoustic factors:", FACTOR_OUTPUT, factor_info)

In [ ]:
# Drive에 고정된 class-stratified 80/10/10 split을 생성하거나 재사용
SPLIT_MANIFEST = Path(
    "/content/drive/MyDrive/OnomaCap/splits/"
    "onomacap_7961_seed20_v1.csv"
)
splits = load_or_create_split_manifest(
    df,
    SPLIT_MANIFEST,
    split_seed=SPLIT_SEED,
)
assert {name: len(frame) for name, frame in splits.items()} == {
    "train": 6_368, "val": 796, "test": 797
}
print({name: len(frame) for name, frame in splits.items()})
print("split seed:", SPLIT_SEED, "model seed:", MODEL_SEED)

In [ ]:
# WavCaps JSON 작성: 각 caption은 공백으로 구분된 canonical Jamo
JSON_DIR = WAVCAPS_DIR / "captioning/data/OnomaCap/json_files"
JSON_DIR.mkdir(parents=True, exist_ok=True)

def to_wavcaps_item(row):
    return {
        "audio": str(Path(row["audio_path"]).resolve()),
        **{
            f"caption_{index}": str(row[f"candidate{index}_jamo"])
            for index in range(1, 6)
        },
    }

maximum_target_tokens = 0
for split_name, frame in splits.items():
    records = [to_wavcaps_item(row) for _, row in frame.iterrows()]
    output_path = JSON_DIR / f"{split_name}.json"
    output_path.write_text(
        json.dumps({"data": records}, ensure_ascii=False, indent=2),
        encoding="utf-8",
    )
    maximum_target_tokens = max(
        maximum_target_tokens,
        max(len(record[f"caption_{index}"].split()) for record in records for index in range(1, 6)),
    )
    print(output_path, len(records))

assert maximum_target_tokens + 2 <= 80  # BOS/EOS 포함
assert len(splits["train"]) * 5 == 31_840
print("maximum Jamo target tokens (without BOS/EOS):", maximum_target_tokens)

In [ ]:
# HTSAT 사전학습 checkpoint 배치
HTSAT_SOURCE = Path("/content/drive/MyDrive/OnomaCap/pretrained/HTSAT.ckpt")
HTSAT_DESTINATION = WAVCAPS_DIR / "captioning/pretrained_models/audio_encoder/HTSAT.ckpt"
if not HTSAT_SOURCE.is_file():
    raise FileNotFoundError(
        f"{HTSAT_SOURCE}가 없습니다. Drive에 HTSAT.ckpt를 먼저 올려주세요."
    )
HTSAT_DESTINATION.parent.mkdir(parents=True, exist_ok=True)
shutil.copy2(HTSAT_SOURCE, HTSAT_DESTINATION)
assert HTSAT_DESTINATION.stat().st_size == HTSAT_SOURCE.stat().st_size
print(HTSAT_DESTINATION)

In [ ]:
# 데이터/자모 문법 기본 검증
sample_jamo = "ᄐ ᅡ ᄃ ᅳ ᄅ ᅳ ᆼ"
assert jamo_to_hangul_caption(sample_jamo) == "타 드 릉"
assert "ᄀ" != "ᆨ"  # 같은 ㄱ 계열이어도 초성과 종성은 별도 codepoint/token

sample_record = json.loads((JSON_DIR / "train.json").read_text(encoding="utf-8"))["data"][0]
for index in range(1, 6):
    assert jamo_to_hangul_caption(sample_record[f"caption_{index}"])
print(sample_record["caption_1"], "->", jamo_to_hangul_caption(sample_record["caption_1"]))

In [ ]:
# 본 학습 전 실제 1배치 BF16 forward/backward + constrained generation smoke test
import ruamel.yaml as yaml
from torch.utils.data import DataLoader, Subset

SMOKE_MARKER = Path("/content/onomacap_jamo_bf16_smoke_passed")
SMOKE_MARKER.unlink(missing_ok=True)
previous_cwd = Path.cwd()
smoke_model = None

try:
    captioning_dir = WAVCAPS_DIR / "captioning"
    os.chdir(captioning_dir)
    if str(captioning_dir) not in sys.path:
        sys.path.insert(0, str(captioning_dir))

    from data_handling.datamodule import AudioCaptionDataModule, collate_fn
    from models.bart_captioning import BartCaptionModel
    from pretrain import validate
    from tools.optim_utils import get_optimizer
    from tools.utils import setup_seed

    with open("settings/onomacap_jamo_factor.yaml", "r") as stream:
        smoke_config = yaml.safe_load(stream)
    smoke_config["data_args"] = dict(smoke_config["data_args"])
    smoke_config["data_args"]["batch_size"] = 2
    smoke_config["data_args"]["num_workers"] = 0
    setup_seed(smoke_config["seed"])

    assert torch.cuda.is_available(), "CUDA GPU가 필요합니다."
    assert torch.cuda.is_bf16_supported(), "BF16 지원 CUDA GPU가 필요합니다."
    smoke_device = "cuda"

    smoke_data = AudioCaptionDataModule(smoke_config, "OnomaCap")
    audio, text, _, _, factors, factor_mask = next(iter(smoke_data.train_dataloader()))
    smoke_model = BartCaptionModel(smoke_config).to(smoke_device)
    assert factors.shape[:2] == (2, 6)
    assert factor_mask.shape == (2, factors.shape[-1])
    assert len(smoke_model.jamo_to_id) == 67
    assert smoke_model.jamo_to_id["ᄀ"] != smoke_model.jamo_to_id["ᆨ"]
    assert all(
        smoke_model.tokenizer.convert_tokens_to_ids(token) == token_id
        for token, token_id in smoke_model.jamo_to_id.items()
    )

    optimizer = get_optimizer(
        smoke_model.parameters(),
        lr=smoke_config["optim_args"]["lr"],
        betas=smoke_config["optim_args"]["betas"],
        eps=smoke_config["optim_args"]["eps"],
        momentum=smoke_config["optim_args"]["momentum"],
        weight_decay=smoke_config["optim_args"]["weight_decay"],
        optimizer_name=smoke_config["optim_args"]["optimizer_name"],
    )
    smoke_phase = smoke_model.set_training_epoch(1)
    expected_factor_only = smoke_config["factor_args"].get("factor_only_epochs", 0) >= 1
    assert smoke_phase["factor_only"] == expected_factor_only
    if expected_factor_only:
        assert smoke_phase["audio_modality_dropout"] == 1.0
        assert smoke_phase["trainable_parameters"] < sum(
            parameter.numel() for parameter in smoke_model.parameters()
        )
    optimizer.zero_grad(set_to_none=True)
    with torch.autocast(device_type="cuda", dtype=torch.bfloat16):
        smoke_loss = smoke_model(
            audio.to(smoke_device),
            text,
            factors=factors.to(smoke_device),
            factor_mask=factor_mask.to(smoke_device),
        )
    assert torch.isfinite(smoke_loss)
    smoke_loss.backward()
    gradient_norm = torch.nn.utils.clip_grad_norm_(
        smoke_model.parameters(), 2.0, error_if_nonfinite=True
    )
    optimizer.step()
    with torch.autocast(device_type="cuda", dtype=torch.bfloat16):
        smoke_attention = smoke_model.teacher_forced_factor_attention(
            audio[:1].to(smoke_device),
            text[:1],
            factors[:1].to(smoke_device),
            factor_mask[:1].to(smoke_device),
        )
    smoke_factor_mass = smoke_attention["factor_attention_mass"].float()
    assert smoke_factor_mass.shape[-1] == 6
    assert torch.isfinite(smoke_factor_mass).all()
    assert torch.allclose(
        smoke_factor_mass.sum(dim=-1),
        torch.ones_like(smoke_factor_mass[..., 0]),
        atol=1e-3,
    )

    smoke_val_loader = DataLoader(
        Subset(smoke_data.val_set, range(2)),
        batch_size=2,
        shuffle=False,
        num_workers=0,
        collate_fn=collate_fn,
    )
    smoke_log_dir = Path("outputs/smoke")
    smoke_log_dir.mkdir(parents=True, exist_ok=True)
    smoke_metrics = validate(
        smoke_val_loader,
        smoke_model,
        smoke_device,
        smoke_log_dir,
        epoch=0,
        beam_size=EVAL_BEAM_SIZE,
    )
    assert set(smoke_metrics) == {"bleu_1", "bleu_2", "bleu_3", "bleu_4", "meteor", "rouge_l"}
    SMOKE_MARKER.write_text("passed", encoding="utf-8")
    print("BF16 smoke loss:", float(smoke_loss.detach().cpu()))
    print("gradient norm:", float(gradient_norm.detach().cpu()))
finally:
    os.chdir(previous_cwd)
    del smoke_model
    gc.collect()
    torch.cuda.empty_cache()

assert SMOKE_MARKER.is_file()

In [ ]:
from pathlib import Path

assert Path("/content/onomacap_jamo_bf16_smoke_passed").is_file()

captioning_dir = WAVCAPS_DIR / "captioning"

%cd "{captioning_dir}"
!PYTHONPATH="{captioning_dir}" \
python train.py \
  --exp_name {EXPERIMENT_NAME} \
  --config settings/onomacap_jamo_factor.yaml \
  --lr 3e-5 \
  --seed {MODEL_SEED}

In [ ]:
# Human oracle benchmark 공통 함수 + Latin benchmark
import numpy as np
from pycocoevalcap.bleu.bleu import Bleu

captioning_dir = WAVCAPS_DIR / "captioning"
if str(captioning_dir) not in sys.path:
    sys.path.insert(0, str(captioning_dir))
from data_handling.text_transform import text_preprocess
from eval_metrics import evaluate_metrics


def oracle_human_benchmark(frame, columns, transform):
    cases = []
    for _, row in frame.iterrows():
        captions = [transform(str(row[column])) for column in columns]
        for held_out, prediction in enumerate(captions):
            cases.append({
                "audio_file": str(row["audio_file"]),
                "candidate_index": held_out,
                "prediction": prediction,
                "references": [caption for index, caption in enumerate(captions) if index != held_out],
            })

    gts = {index: case["references"] for index, case in enumerate(cases)}
    res = {index: [case["prediction"]] for index, case in enumerate(cases)}
    _, per_candidate_bleu = Bleu(4).compute_score(gts, res)
    bleu_1 = np.asarray(per_candidate_bleu[0])

    predictions, references, selected = [], [], []
    for offset in range(0, len(cases), 5):
        local_index = int(np.argmax(bleu_1[offset:offset + 5]))
        case = cases[offset + local_index]
        selected.append(case["candidate_index"] + 1)
        predictions.append({
            "file_name": case["audio_file"],
            "caption_predicted": case["prediction"],
        })
        references.append({
            "file_name": case["audio_file"],
            **{
                f"caption_{index}": caption
                for index, caption in enumerate(case["references"], start=1)
            },
        })

    metrics = evaluate_metrics(predictions, references, nb_reference_captions=4)
    return {name: float(values["score"]) for name, values in metrics.items()}, selected


oracle_scores, _ = oracle_human_benchmark(
    splits["test"],
    [f"candidate{index}_en" for index in range(1, 6)],
    text_preprocess,
)
display(oracle_scores)


In [ ]:
# 동일한 공통 함수로 Jamo human oracle benchmark
jamo_oracle_scores, selected_candidate_indices = oracle_human_benchmark(
    splits["test"],
    [f"candidate{index}_jamo" for index in range(1, 6)],
    jamo_to_hangul_caption,
)
display(jamo_oracle_scores)
display({index: selected_candidate_indices.count(index) for index in range(1, 6)})


In [ ]:
# best validation checkpoint 확인
FOLDER_NAME = (
    f"{EXPERIMENT_NAME}_lr_3e-05_batch_16_seed_{MODEL_SEED}"
)
CHECKPOINT_DIR = Path("/content/drive/MyDrive/OnomaCap/checkpoints") / FOLDER_NAME
TEST_EPOCH = 11
CHECKPOINT_PATH = CHECKPOINT_DIR / "epochs" / f"epoch_{TEST_EPOCH:02d}.pt"
# BEST_PATH = CHECKPOINT_DIR / "best_model.pt"
# best = torch.load(BEST_PATH, map_location="cpu", weights_only=False)
# print("best epoch:", best["epoch"])
# print("selection metric:", best["selection_metric"])
# print("best val BLEU-1:", best["selection_score"])
# print("all val scores:", best["val_scores"])

In [ ]:
# 모든 epoch를 공통 beam=3으로 재평가하고 joint BLEU-1 best를 test
import os
import gc
import torch
import pandas as pd
import ruamel.yaml as yaml
from loguru import logger

BEAM3_RECHECK_SIZE = EVAL_BEAM_SIZE
beam3_previous_cwd = Path.cwd()
beam3_model = None

try:
    captioning_dir = WAVCAPS_DIR / "captioning"
    os.chdir(captioning_dir)
    if str(captioning_dir) not in sys.path:
        sys.path.insert(0, str(captioning_dir))

    from data_handling.datamodule import AudioCaptionDataModule
    from models.bart_captioning import BartCaptionModel
    from pretrain import validate

    with open("settings/onomacap_jamo_factor.yaml", "r") as stream:
        beam3_config = yaml.safe_load(stream)
    beam3_data = AudioCaptionDataModule(beam3_config, "OnomaCap")
    beam3_model = BartCaptionModel(beam3_config).to("cuda")
    epoch_paths = sorted(
        (CHECKPOINT_DIR / "epochs").glob("epoch_*.pt"),
        key=lambda path: int(path.stem.removeprefix("epoch_")),
    )
    if not epoch_paths:
        raise FileNotFoundError(f"No epoch checkpoints: {CHECKPOINT_DIR / 'epochs'}")

    beam3_root = Path(beam3_config["results"]["root"]) / FOLDER_NAME / "beam3_recheck"
    beam3_root.mkdir(parents=True, exist_ok=True)
    logger.remove()
    beam3_rows = []

    for epoch_path in epoch_paths:
        epoch = int(epoch_path.stem.removeprefix("epoch_"))
        checkpoint = torch.load(epoch_path, map_location="cuda", weights_only=False)
        beam3_model.load_state_dict(checkpoint["model"])
        row = {"epoch": epoch, "checkpoint": str(epoch_path)}

        for condition, disable_audio in (("factor_only", True), ("joint", False)):
            metrics = validate(
                beam3_data.val_dataloader(),
                beam3_model,
                device="cuda",
                log_dir=beam3_root / "val" / condition,
                epoch=epoch,
                beam_size=BEAM3_RECHECK_SIZE,
                disable_audio=disable_audio,
                condition=f"{condition}_beam3",
            )
            row.update({
                f"{condition}/{name}": float(values["score"])
                for name, values in metrics.items()
            })

        beam3_rows.append(row)
        print(
            f"epoch {epoch:02d} | factor-only BLEU-1 "
            f"{row['factor_only/bleu_1']:.4f} | joint BLEU-1 {row['joint/bleu_1']:.4f}"
        )

    beam3_scores = pd.DataFrame(beam3_rows).sort_values("epoch").reset_index(drop=True)
    beam3_csv = beam3_root / "beam3_validation_scores.csv"
    beam3_scores.to_csv(beam3_csv, index=False)
    display(beam3_scores)

    best_row = beam3_scores.loc[beam3_scores["joint/bleu_1"].idxmax()]
    best_beam3_epoch = int(best_row["epoch"])
    best_beam3_path = Path(best_row["checkpoint"])
    beam3_model.load_state_dict(
        torch.load(best_beam3_path, map_location="cuda", weights_only=False)["model"]
    )
    beam3_test_metrics = validate(
        beam3_data.test_dataloader(),
        beam3_model,
        device="cuda",
        log_dir=beam3_root / "test" / "joint",
        epoch=best_beam3_epoch,
        beam_size=BEAM3_RECHECK_SIZE,
        disable_audio=False,
        condition="joint_beam3_test",
    )
    print("best beam-3 epoch:", best_beam3_epoch)
    print("validation table:", beam3_csv)
    display({name: float(values["score"]) for name, values in beam3_test_metrics.items()})
finally:
    os.chdir(beam3_previous_cwd)
    if beam3_model is not None:
        del beam3_model
    gc.collect()
    torch.cuda.empty_cache()


In [ ]:
# 지정 epoch: 한글 평가와 로마자 평가 모두 공통 beam=3
TEST_EPOCH = 12
TEST_BEAM_SIZE = EVAL_BEAM_SIZE

test_previous_cwd = Path.cwd()
test_model = None
try:
    captioning_dir = WAVCAPS_DIR / "captioning"
    os.chdir(captioning_dir)
    if str(captioning_dir) not in sys.path:
        sys.path.insert(0, str(captioning_dir))

    from data_handling.datamodule import AudioCaptionDataModule
    from models.bart_captioning import BartCaptionModel
    from pretrain import validate
    from loguru import logger
    logger.remove()
    logger.add(sys.stdout, filter=lambda record: record["extra"].get("indent") == 1)

    with open("settings/onomacap_jamo_factor.yaml", "r") as stream:
        test_config = yaml.safe_load(stream)
    test_data = AudioCaptionDataModule(test_config, "OnomaCap")
    test_model = BartCaptionModel(test_config).to("cuda")
    checkpoint_path = CHECKPOINT_DIR / "epochs" / f"epoch_{TEST_EPOCH:02d}.pt"
    checkpoint = torch.load(checkpoint_path, map_location="cuda", weights_only=False)
    test_model.load_state_dict(checkpoint["model"])

    test_log_dir = Path(test_config["results"]["root"]) / FOLDER_NAME / f"epoch_{TEST_EPOCH:02d}"
    english_references = {
        str(row["audio_file"]): [
            normalize_romanized_caption(row[f"candidate{index}_en"])
            for index in range(1, 6)
        ]
        for _, row in splits["test"].iterrows()
    }
    validation_args = dict(
        data_loader=test_data.test_dataloader(),
        model=test_model,
        device="cuda",
        epoch=TEST_EPOCH,
        beam_size=TEST_BEAM_SIZE,
    )
    korean_metrics = validate(log_dir=test_log_dir, **validation_args)
    romanized_metrics = validate(
        log_dir=test_log_dir / "romanized",
        condition="romanized",
        prediction_transform=hangul_caption_to_romanized,
        reference_captions_by_file=english_references,
        **validation_args,
    )

    score_comparison = pd.DataFrame({
        "korean": {name: float(values["score"]) for name, values in korean_metrics.items()},
        "romanized": {name: float(values["score"]) for name, values in romanized_metrics.items()},
    })
    score_comparison["delta"] = score_comparison["romanized"] - score_comparison["korean"]
    display(score_comparison)
finally:
    os.chdir(test_previous_cwd)
    if test_model is not None:
        del test_model
    gc.collect()
    torch.cuda.empty_cache()


In [ ]:
import importlib
import matplotlib
import seaborn

importlib.reload(matplotlib)
importlib.reload(seaborn)

import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
# test set teacher forcing: 자모별 factor attention 집계
!apt-get -qq update
!apt-get -qq install fonts-nanum

from factor_attention_analysis import FactorAttentionAnalyzer, configure_korean_plots

font_name = configure_korean_plots()
token_groups = {
    "choseong": CHOSEONG,
    "jungseong": JUNGSEONG,
    "jongseong": JONGSEONG,
}
display_groups = {
    "choseong": tuple("ㄱㄲㄴㄷㄸㄹㅁㅂㅃㅅㅆㅇㅈㅉㅊㅋㅌㅍㅎ"),
    "jungseong": tuple("ㅏㅐㅑㅒㅓㅔㅕㅖㅗㅘㅙㅚㅛㅜㅝㅞㅟㅠㅡㅢㅣ"),
    "jongseong": tuple("ㄱㄲㄳㄴㄵㄶㄷㄹㄺㄻㄼㄽㄾㄿㅀㅁㅂㅄㅅㅆㅇㅈㅊㅋㅌㅍㅎ"),
}
attention_output_dir = (
    Path("/content/drive/MyDrive/OnomaCap/results")
    / FOLDER_NAME
    / "factor_attention"
)
attention_analyzer = FactorAttentionAnalyzer(
    captioning_dir=WAVCAPS_DIR / "captioning",
    config_path=factor_config_destination,
    checkpoint_path=CHECKPOINT_PATH,
    token_groups=token_groups,
    display_groups=display_groups,
)
attention_table = attention_analyzer.aggregate(
    attention_output_dir / "teacher_forced_jamo_factor_attention.csv"
)
display(attention_table)
attention_analyzer.plot_category_heatmaps(attention_table, attention_output_dir)


In [ ]:
# 선택한 자모의 factor × 상대시간 attention

def plot_jamo_factor_time_attention(category, jamo, **kwargs):
    return attention_analyzer.plot_time(category, jamo, **kwargs)


def clear_factor_time_attention_cache():
    attention_analyzer.clear()


# 예시:
# plot_jamo_factor_time_attention(category="choseong", jamo="ㅅ")
# plot_jamo_factor_time_attention(category="jongseong", jamo="ㅅ", reference_index=0)


In [ ]:
plot_jamo_factor_time_attention(category='jongseong', jamo="ㄱ")